# Super Mario Bros Gameplay Training with ConvNeXt-RWKV7 Gamepad

Train the **ConvNeXt-RWKV7 Gamepad** model on the **Super Mario Bros World Model** dataset (`DylanRiden/smb-worldmodel-data`) using **PyTorch Lightning** on Kaggle **2x T4 GPUs** (DDP with mixed precision).

### Pipeline Architecture
1. **Input Normalization:** `_InputNormalize` (DINOv3 ImageNet mean/std)
2. **Spatial Pooling:** `AdaptiveLearnedPool2d` (downsamples inputs to 224x224)
3. **Vision Backbone:** `ConvNeXt-Tiny` with pre-trained **DINOv3** weights (`facebook/dinov3-convnext-tiny-pretrain-lvd1689m`)
4. **Spatial Aggregation:** `LearnedWeightedGAP` (spatial attention + global average pooling)
5. **Temporal Convolution:** `CausalConv1d` with residual shortcut
6. **Recurrent Reasoning:** 4x `RWKV-7` (Goose) linear attention blocks
7. **Gamepad Head:** 21-D output (17 boolean buttons via BCE loss + 4 joystick axes in [-1.0, 1.0] via MSE loss)


In [ ]:
# Install the ConvNeXt Platform package directly from GitHub
%pip install -q "git+https://github.com/Gabz4200/ConvNeXt_Platform.git"


In [ ]:
import os
from functools import partial
import torch
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, RichProgressBar
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

# Allow safe unpickling for PyTorch 2.6+
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

# Authenticate with Hugging Face Hub using Kaggle secret 'HF_TOKEN' for DINOv3 access
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token)
    print("Successfully authenticated with Hugging Face Hub via Kaggle secret (HF_TOKEN)!")
except Exception as e:
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print("Authenticated with Hugging Face Hub via environment variable HF_TOKEN.")
    else:
        print(f"Hugging Face Hub authentication notice: {e}")

# Set global seed for reproducibility
L.seed_everything(3407, workers=True)

# Check hardware accelerators (2x T4 on Kaggle)
num_gpus = torch.cuda.device_count()
print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")
print(f"Available GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
from src.data.smb_datamodule import SMBDataModule

# DataModule hyperparameters (Super Mario Bros Dataset)
datamodule = SMBDataModule(
    # Storage and source
    data_dir="data/smb",                           # Local directory to store and extract dataset frames
    repo_id="DylanRiden/smb-worldmodel-data",       # Hugging Face Hub dataset repo ID
    filename="smb_frames.zip",                     # Archive filename to download from Hugging Face Hub
    download=True,                                 # Automatically download and extract archive if missing
    # Batching and worker resources
    batch_size=64,                                 # Per-GPU batch size (effective batch size = 128 on 2x T4)
    num_workers=2,                                 # DataLoader background subprocesses per GPU (optimal for 4-vCPU Kaggle)
    pin_memory=torch.cuda.is_available(),          # Pin memory tensors in page-locked CUDA RAM for faster DMA transfers
    # Splitting and sample budget
    val_ratio=0.1,                                 # Fraction of dataset files reserved for validation (10%)
    test_ratio=0.1,                                # Fraction of dataset files reserved for test evaluation (10%)
    max_samples=None,                              # Optional maximum sample limit for rapid debugging (None = full dataset)
    # Image and target mapping
    image_size=(224, 224),                         # Input image spatial resolution (height, width)
    target_mode="gamepad_21",                      # Target action representation: 'gamepad_21' (21-D) or 'nes_8' (8-D)
    seed=3407,                                     # Random seed for deterministic and reproducible dataset splits
)

# Download and extract dataset before multi-GPU DDP spawning
datamodule.prepare_data()
datamodule.setup("fit")
print(f"Training samples:   {len(datamodule.data_train)}")
print(f"Validation samples: {len(datamodule.data_val)}")


In [ ]:
from src.models.components.convnext_rwkv7 import ConvNeXtRWKV7Gamepad
from src.models.convnext_rwkv7_module import ConvNeXtRWKV7GamepadLitModule

# Model architecture hyperparameters (ConvNeXt-RWKV7 Gamepad)
net = ConvNeXtRWKV7Gamepad(
    # Input and spatial preprocessing
    in_chans=3,                                    # Number of input image channels (3 for RGB gameplay frames)
    pool_intermediate_features=32,                 # Intermediate channels inside AdaptiveLearnedPool2d downsampler
    # ConvNeXt vision backbone
    convnext_size="tiny",                          # ConvNeXt scale ('tiny', 'small', 'base', 'large')
    convnext_dims=None,                            # Custom stage channel widths (None uses convnext_size defaults: [96, 192, 384, 768])
    convnext_depths=None,                          # Custom stage block depths (None uses convnext_size defaults: [3, 3, 9, 3])
    convnext_drop_path_rate=0.0,                   # Stochastic depth / DropPath rate for ConvNeXt residual branches
    convnext_layer_scale_init_value=1e-6,          # Initial multiplier for LayerScale in ConvNeXt blocks
    pretrained_dinov3=True,                        # Load self-supervised DINOv3 pre-trained visual representations
    dinov3_repo_id="facebook/dinov3-convnext-tiny-pretrain-lvd1689m", # Hugging Face Hub repository ID for DINOv3 weights
    bypass_stem=False,                             # If True, pooler feeds stage-0 directly (56x56) bypassing ConvNeXt stem
    freeze_convnext=True,                          # Freeze ConvNeXt weights while preserving autograd flow to AdaptiveLearnedPool2d
    # Spatial feature aggregation
    gap_kernel_size=3,                             # Kernel size for spatial attention 2D convolution in LearnedWeightedGAP
    gap_concat=True,                               # Whether LearnedWeightedGAP concatenates uniform GAP + weighted GAP features
    # Temporal and recurrent dynamics
    causal_conv_kernel_size=3,                     # 1D causal convolution kernel size along temporal sequence dimension
    rwkv_dim=256,                                  # Hidden representation channel width across RWKV-7 recurrent blocks
    rwkv_head_size=64,                             # Attention head dimension for RWKV-7 linear attention
    rwkv_layers=4,                                 # Number of stacked RWKV-7 (Goose) linear attention recurrent blocks
    rwkv_dim_ffn=None,                             # RWKV-7 Feed-Forward Network hidden dimension (None = 4 * rwkv_dim = 1024)
    # Gamepad prediction head
    head_hidden_dim=256,                           # Hidden projection layer width inside GamepadHead
    num_buttons=17,                                # Number of discrete gamepad button logits (BCE loss)
    num_joysticks=2,                               # Number of dual-axis analog joysticks (2 joysticks * 2 axes = 4 MSE values in [-1, 1])
)

# Optimizer and learning rate scheduler hyperparameters
max_epochs = 15

# AdamW optimizer factory (trains pooler, spatial GAP, causal conv, RWKV-7 blocks, and gamepad head)
optimizer_factory = partial(
    torch.optim.AdamW,
    lr=1e-3,                                       # Base learning rate for trainable model components
    weight_decay=0.01,                             # Decoupled L2 weight decay regularization
    betas=(0.9, 0.999),                            # Adam first and second momentum coefficient estimates
    eps=1e-8,                                      # Epsilon term for numerical stability in optimizer denominator
)

# Cosine annealing learning rate scheduler factory
scheduler_factory = partial(
    torch.optim.lr_scheduler.CosineAnnealingLR,
    T_max=max_epochs,                              # Maximum number of epochs for full cosine decay cycle
    eta_min=1e-6,                                  # Minimum learning rate floor at the end of the cosine schedule
)

# PyTorch Lightning module wrapper
model = ConvNeXtRWKV7GamepadLitModule(
    net=net,                                       # Instantiated ConvNeXtRWKV7Gamepad backbone network
    optimizer=optimizer_factory,                   # Partial optimizer callable
    scheduler=scheduler_factory,                   # Partial learning rate scheduler callable
    joystick_loss_weight=1.0,                      # Scalar loss multiplier weighting analog joystick MSE vs BCE button loss
    convnext_lr=None,                              # Differential LR for ConvNeXt when unfrozen (e.g. 1e-5; None = base lr)
    compile=False,                                 # Whether to JIT-compile model backbone with torch.compile
)


In [ ]:
# Hardware strategy and precision configuration
strategy = "ddp_notebook" if num_gpus > 1 else "auto"  # Distributed training strategy ('ddp_notebook' prevents fork crashes on Kaggle)
devices = num_gpus if num_gpus > 0 else "auto"         # Number of GPU devices to allocate
accelerator = "gpu" if num_gpus > 0 else "cpu"         # Hardware accelerator ('gpu', 'cpu', or 'auto')
precision = "16-mixed" if num_gpus > 0 else "32-true"  # Precision mode ('16-mixed' for FP16 Tensor Cores, '32-true' for FP32)

# Logging configuration
logger = CSVLogger(
    save_dir="logs",                                   # Base directory for logging metrics and CSV output
    name="smb_gamepad",                                # Experiment subdirectory name
)

# Training callbacks
callbacks = [
    ModelCheckpoint(
        dirpath="checkpoints/smb_gamepad",             # Directory where model weight checkpoints will be stored
        filename="smb-{epoch:02d}-{val/loss:.4f}",     # Checkpoint filename pattern with epoch and metric tags
        monitor="val/loss",                            # Metric key to track for best checkpoint selection
        mode="min",                                    # Monitored optimization mode ('min' for loss, 'max' for accuracy)
        save_top_k=2,                                  # Number of best checkpoints to retain on disk
        save_last=True,                                # Always keep the last checkpoint (last.ckpt) for resume support
    ),
    EarlyStopping(
        monitor="val/loss",                            # Metric key to track for early stopping
        patience=4,                                    # Number of validation checks with no improvement before stopping
        min_delta=1e-4,                                # Minimum change in monitored metric to qualify as improvement
        mode="min",                                    # Optimization direction for monitored metric
    ),
    RichProgressBar(),                                 # Clean progress bar rendering with ETA and metrics
]

# PyTorch Lightning trainer
trainer = L.Trainer(
    # Hardware and distributed
    accelerator=accelerator,                           # 'gpu' or 'cpu'
    devices=devices,                                   # Device count
    strategy=strategy,                                 # 'ddp_notebook' for multi-GPU Kaggle
    precision=precision,                               # Mixed precision training mode
    # Epochs and step budgets
    max_epochs=max_epochs,                             # Maximum total training epochs
    max_steps=-1,                                      # Total step budget limit (-1 runs for full max_epochs)
    # Optimization and regularization
    gradient_clip_val=1.0,                             # Maximum gradient norm for clipping (stabilizes recurrent RWKV-7 training)
    accumulate_grad_batches=1,                         # Gradient accumulation step count (simulates larger batch sizes)
    # Logging and validation frequency
    callbacks=callbacks,                               # List of instantiated Lightning callbacks
    logger=logger,                                     # Experiment metric logger
    log_every_n_steps=50,                              # Step interval for training metric logging
    val_check_interval=1.0,                            # Validation loop frequency (1.0 = once per epoch)
    # Debugging and reproducibility controls
    fast_dev_run=False,                                # Set to True or 1 to run 1 batch sanity check through train/val/test
    deterministic=False,                               # Set to True to enforce deterministic PyTorch operations
)


In [ ]:
# Fit the model on the Super Mario Bros dataset
trainer.fit(model=model, datamodule=datamodule)

# Test evaluation using best checkpoint
trainer.test(model=model, datamodule=datamodule, ckpt_path="best")


In [ ]:
# Real-Time Online Recurrent Streaming Demo (Frame-by-Frame Inference)
model.eval()
device = next(model.parameters()).device

# Initialize persistent streaming state
streaming_state = model.net.init_streaming_state(batch_size=1, device=device)

# Load test frame from datamodule
datamodule.setup("test")
test_frame, target_gamepad = datamodule.data_test[0]
input_tensor = test_frame.unsqueeze(0).to(device)  # Shape: (1, 3, 224, 224)

# Execute O(1) recurrent step update
with torch.no_grad():
    (full_gamepad, buttons_logits, joysticks), streaming_state = model.net.step(input_tensor, streaming_state)

btn_probs = buttons_logits.sigmoid().squeeze(0).cpu().tolist()
joy_axes = joysticks.squeeze(0).cpu().tolist()

print(f"Predicted 21-D Gamepad Vector Shape: {full_gamepad.shape}")
print("Button Probabilities (first 8 mapped):", [round(p, 3) for p in btn_probs[:8]])
print(f"Predicted Left Stick (X, Y): ({joy_axes[0]:.3f}, {joy_axes[1]:.3f})")
print(f"Target Action Vector (21-D): {target_gamepad.shape}")
